In [ ]:
import time
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
import random

# Function to compute the square
def square(n):
    return n * n

# List of random numbers
numbers = [random.randint(1, 100) for _ in range(10**7)]

# Sequential processing
def sequential_square(numbers):
    return [square(n) for n in numbers]

# Multiprocessing with separate processes
def multiprocessing_square(numbers):
    processes = []
    results = []
    
    for number in numbers:
        process = multiprocessing.Process(target=lambda n: results.append(square(n)), args=(number,))
        processes.append(process)
        process.start()
    
    for process in processes:
        process.join()

# Pool with map
def pool_map_square(numbers):
    with multiprocessing.Pool() as pool:
        return pool.map(square, numbers)

# Pool with apply_async
def pool_apply_square(numbers):
    with multiprocessing.Pool() as pool:
        results = [pool.apply_async(square, (n,)) for n in numbers]
        return [result.get() for result in results]

# Using ProcessPoolExecutor
def process_pool_square(numbers):
    with ProcessPoolExecutor() as executor:
        return list(executor.map(square, numbers))

# Timing each function
start = time.time()
sequential_square(numbers)
print("Sequential:", time.time() - start)

start = time.time()
multiprocessing_square(numbers)
print("Multiprocessing:", time.time() - start)

start = time.time()
pool_map_square(numbers)
print("Pool map:", time.time() - start)

start = time.time()
pool_apply_square(numbers)
print("Pool apply_async:", time.time() - start)

start = time.time()
process_pool_square(numbers)
print("ProcessPoolExecutor:", time.time() - start)

Sequential: 0.5827322006225586


In [2]:
import multiprocessing
import time
import random

class ConnectionPool:
    def __init__(self, size):
        self.size = size
        self.connections = [f"Connection-{i}" for i in range(size)]
        self.semaphore = multiprocessing.Semaphore(size)

    def get_connection(self):
        self.semaphore.acquire()
        return self.connections.pop()

    def release_connection(self, connection):
        self.connections.append(connection)
        self.semaphore.release()

def access_database(pool, process_id):
    print(f"Process-{process_id} waiting for a connection...")
    connection = pool.get_connection()
    print(f"Process-{process_id} acquired {connection}")
    time.sleep(random.uniform(0.5, 2))
    pool.release_connection(connection)
    print(f"Process-{process_id} released {connection}")

if __name__ == "__main__":
    pool = ConnectionPool(size=3)
    processes = []

    for i in range(5):
        process = multiprocessing.Process(target=access_database, args=(pool, i))
        processes.append(process)
        process.start()

    for process in processes:
        process.join()


Process-0 waiting for a connection...
Process-1 waiting for a connection...Process-0 acquired Connection-2
Process-3 waiting for a connection...Process-2 waiting for a connection...Process-4 waiting for a connection...



Process-1 acquired Connection-2Process-4 acquired Connection-2

Process-0 released Connection-2Process-3 acquired Connection-2

Process-4 released Connection-2Process-2 acquired Connection-2

Process-1 released Connection-2
Process-3 released Connection-2
Process-2 released Connection-2
